# XGBoost Classification for TLS Profiling

This notebook evaluates a supervised XGBoost baseline on extracted TLS traffic features. Unlike the anomaly-detection baselines, this model is trained to classify each flow directly into one of three categories: `system`, `malware`, or `unknown`.


In [1]:
%pip install xgboost shap

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.7/131.7 MB 39.5 MB/s  0:00:03m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 32.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 293.6/293.6 MB 47.5 MB/s  0:00:05m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [shap]4/5 [shap]st]kle]cu12]
Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys
sys.path.append('../../src')

## Feature Groups

- `FLOW`: basic bidirectional flow statistics such as bytes, packets, rates, and duration (`bs`, `ps`, `br`, `pr`, `td`)
- `CTLS`: client-side TLS metadata (`tls.cver`, `tls.ccs`, `tls.cext`, `tls.csg`, `tls.alpn`, `tls.csv`)
- `STLS`: server-side TLS metadata (`tls.sver`, `tls.scs`, `tls.sext`, `tls.ssv`)
- `REC`: ordered sequence of signed TLS record lengths (`tls.rec`, first 20 records)

The notebook evaluates each group individually and all group combinations to understand which feature families are most useful for supervised multiclass classification.


In [3]:
FLOW = ["bs","ps", "br", "pr", "td"]
CTLS = ["tls.cver", "tls.ccs", "tls.cext", "tls.csg", "tls.alpn", "tls.csv"]
STLS = ["tls.sver", "tls.scs", "tls.sext", "tls.ssv"]
REC = ["tls.rec"]

In [4]:
from pathlib import Path

import pandas as pd
from tls_profiling.preprocessing import build_and_fit_preprocessor

data_dir = Path("../data-ext")
#data_dir = Path("../data")

print(f"Loading extracted features from {data_dir}.")
df_train = pd.read_parquet(data_dir / "malware_train.parquet")
df_val = pd.read_parquet(data_dir / "malware_val.parquet")
df_test = pd.read_parquet(data_dir / "malware_test.parquet")
df_train_label = pd.read_parquet(data_dir / "malware_train_labels.parquet")
df_val_label = pd.read_parquet(data_dir / "malware_val_labels.parquet")
df_test_label = pd.read_parquet(data_dir / "malware_test_labels.parquet")

preprocessor = build_and_fit_preprocessor(df_train)
print("Built preprocessor from df_train.")


Loading extracted features from ../data-ext.
Built preprocessor from df_train.


In [6]:
from tls_profiling.baselines.model_xgboost import XGBoostBaseline, Config
import numpy as np

from sklearn.metrics import average_precision_score, f1_score, roc_auc_score
from sklearn.preprocessing import label_binarize

CLASS_NAMES = ["system", "malware", "unknown"]
CLASS_TO_INDEX = {label: idx for idx, label in enumerate(CLASS_NAMES)}
INDEX_TO_CLASS = {idx: label for label, idx in CLASS_TO_INDEX.items()}

XGB_CONFIG_CANDIDATES = {
    "balanced_hist_deep": Config(
        n_estimators=400,
        max_depth=8,
        learning_rate=0.05,
        min_child_weight=1.0,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        max_train_samples=100_000,
    ),
    "balanced_hist_shallow": Config(
        n_estimators=300,
        max_depth=5,
        learning_rate=0.08,
        min_child_weight=2.0,
        subsample=0.8,
        colsample_bytree=0.7,
        reg_lambda=1.0,
        max_train_samples=100_000,
    ),
}

def encode_labels(series: pd.Series) -> np.ndarray:
    return series.map(CLASS_TO_INDEX).to_numpy(dtype=np.int64)

def decode_labels(values: np.ndarray) -> list[str]:
    return [INDEX_TO_CLASS[int(value)] for value in values]

def select_feature_set(x, feature_set):
    if not feature_set:
        return x

    selected_columns = [
        column for column in x.columns
        if any(column.startswith(prefix) for prefix in feature_set)
    ]
    return x.loc[:, selected_columns]

def compute_multiclass_scores(y_true, y_pred, y_score):
    y_true_bin = label_binarize(y_true, classes=np.arange(len(CLASS_NAMES)))
    per_class_f1 = f1_score(
        y_true,
        y_pred,
        average=None,
        labels=np.arange(len(CLASS_NAMES)),
    )

    return {
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
        "macro_auc": roc_auc_score(
            y_true,
            y_score,
            labels=np.arange(len(CLASS_NAMES)),
            multi_class="ovr",
            average="macro",
        ),
        "macro_ap": average_precision_score(y_true_bin, y_score, average="macro"),
        "system_f1": per_class_f1[CLASS_TO_INDEX["system"]],
        "malware_f1": per_class_f1[CLASS_TO_INDEX["malware"]],
        "unknown_f1": per_class_f1[CLASS_TO_INDEX["unknown"]],
    }

def choose_best_model(x_train, y_train, x_val, y_val):
    best_name = None
    best_model = None
    best_val_macro_f1 = -np.inf

    for config_name, config in XGB_CONFIG_CANDIDATES.items():
        model = XGBoostBaseline(config)
        model.fit(x_train, y_train)

        val_pred = model.predict(x_val)
        val_macro_f1 = f1_score(y_val, val_pred, average="macro")

        if val_macro_f1 > best_val_macro_f1:
            best_name = config_name
            best_model = model
            best_val_macro_f1 = val_macro_f1

    return best_name, best_model, float(best_val_macro_f1)

def evaluate(feature_set):
    y_train = encode_labels(df_train_label["category"])
    y_val = encode_labels(df_val_label["category"])
    y_test = encode_labels(df_test_label["category"])

    x_train_transformed = select_feature_set(preprocessor.transform(df_train), feature_set)
    x_val_transformed = select_feature_set(preprocessor.transform(df_val), feature_set)
    x_test_transformed = select_feature_set(preprocessor.transform(df_test), feature_set)

    best_config_name, model, val_macro_f1 = choose_best_model(
        x_train_transformed,
        y_train,
        x_val_transformed,
        y_val,
    )

    y_pred = model.predict(x_test_transformed)
    y_score = model.predict_proba(x_test_transformed)

    return y_test, y_pred, y_score, val_macro_f1, best_config_name

def debug_csv(feature_set, y_test, y_pred, y_score):
    feature_set_name = "_".join(feature_set)
    output_path = f"tmp/xgb_{feature_set_name}.csv"

    pd.DataFrame({
        "y_test": decode_labels(y_test),
        "y_pred": decode_labels(y_pred),
        "p_system": y_score[:, CLASS_TO_INDEX["system"]],
        "p_malware": y_score[:, CLASS_TO_INDEX["malware"]],
        "p_unknown": y_score[:, CLASS_TO_INDEX["unknown"]],
    }).to_csv(output_path, index=False)

def compute_scores(feature_set):
    y_test, y_pred, y_score, val_macro_f1, best_config_name = evaluate(feature_set)
    scores = compute_multiclass_scores(y_test, y_pred, y_score)

    debug_csv(feature_set, y_test, y_pred, y_score)
    return {
        "best_config": best_config_name,
        "val_macro_f1": val_macro_f1,
        **scores,
    }


## Evaluation

This baseline uses the same train, validation, and test splits as the anomaly-detection notebooks, but the task is now multiclass classification rather than one-class profiling.

Protocol:

- `train`: used to fit supervised XGBoost models
- `validation`: used to choose the better of two XGBoost configurations using `macro F1`
- `test`: held out for final reporting only

Target classes:

- `system`
- `malware`
- `unknown`

Reported metrics:

- `MacroF1`: the main score for the multiclass classifier, giving each class equal weight
- `MacroAucScore`: one-vs-rest macro ROC-AUC computed from predicted class probabilities
- `MacroApScore`: one-vs-rest macro average precision computed from predicted class probabilities
- per-class `F1` columns for `system`, `malware`, and `unknown`

Practical notes:

- this is a supervised classifier, so it is not directly comparable to one-class anomaly detectors in terms of training assumptions
- to keep the full feature-ablation study tractable, each XGBoost fit uses a stratified cap of `100,000` training flows


In [7]:
from itertools import combinations

feature_groups = {
    "FLOW": FLOW,
    "CTLS": CTLS,
    "STLS": STLS,
    "REC": REC,
}

rows = []

group_names = list(feature_groups)
for size in range(1, len(group_names) + 1):
    for group_combo in combinations(group_names, size):
        selected_features = []
        for group_name in group_combo:
            selected_features.extend(feature_groups[group_name])

        feature_set_name = ", ".join(group_combo)
        scores = compute_scores(selected_features)

        rows.append({
            "FeatureSet": feature_set_name,
            "BestConfig": scores["best_config"],
            "ValMacroF1": scores["val_macro_f1"],
            "MacroF1": scores["macro_f1"],
            "MacroAucScore": scores["macro_auc"],
            "MacroApScore": scores["macro_ap"],
            "SystemF1": scores["system_f1"],
            "MalwareF1": scores["malware_f1"],
            "UnknownF1": scores["unknown_f1"],
        })

results_df = pd.DataFrame(
    rows,
    columns=[
        "FeatureSet",
        "BestConfig",
        "ValMacroF1",
        "MacroF1",
        "MacroAucScore",
        "MacroApScore",
        "SystemF1",
        "MalwareF1",
        "UnknownF1",
    ],
)
display(results_df)


,FeatureSet,BestConfig,ValMacroF1,MacroF1,MacroAucScore,MacroApScore,SystemF1,MalwareF1,UnknownF1
0,FLOW,balanced_hist_deep,0.778341,0.783665,0.959993,0.901198,0.952473,0.819612,0.578911
1,CTLS,balanced_hist_deep,0.641192,0.626853,0.803784,0.642050,0.886378,0.743945,0.250234
2,STLS,balanced_hist_deep,0.721921,0.650686,0.923861,0.840674,0.956935,0.589893,0.405231
3,REC,balanced_hist_deep,0.794656,0.787445,0.963088,0.909669,0.967334,0.816632,0.578369
4,"FLOW, CTLS",balanced_hist_deep,0.791323,0.778054,0.962151,0.907716,0.961592,0.812584,0.559985
5,"FLOW, STLS",balanced_hist_deep,0.789942,0.786397,0.964687,0.911116,0.964180,0.821005,0.574005
6,"FLOW, REC",balanced_hist_deep,0.795551,0.800526,0.966993,0.916310,0.967032,0.837176,0.597370
7,"CTLS, STLS",balanced_hist_shallow,0.734974,0.735219,0.930439,0.854714,0.956811,0.769480,0.479366
8,"CTLS, REC",balanced_hist_deep,0.797834,0.797273,0.966355,0.918754,0.967001,0.832641,0.592178
9,"STLS, REC",balanced_hist_deep,0.790000,0.787634,0.963949,0.911890,0.967961,0.815763,0.579179


## Overall Results

The table below ranks feature sets by `MacroF1`, which is the primary metric for this multiclass baseline. `MacroAucScore` and `MacroApScore` are included to make the supervised baseline easier to compare at a high level with the anomaly-detection notebooks.


In [8]:
overall_df = results_df.sort_values("MacroF1", ascending=False)
display(overall_df)

,FeatureSet,BestConfig,ValMacroF1,MacroF1,MacroAucScore,MacroApScore,SystemF1,MalwareF1,UnknownF1
14,"FLOW, CTLS, STLS, REC",balanced_hist_deep,0.804776,0.824986,0.971410,0.927685,0.966628,0.864503,0.643828
11,"FLOW, CTLS, REC",balanced_hist_deep,0.804478,0.824456,0.969970,0.925409,0.967575,0.865118,0.640675
6,"FLOW, REC",balanced_hist_deep,0.795551,0.800526,0.966993,0.916310,0.967032,0.837176,0.597370
12,"FLOW, STLS, REC",balanced_hist_deep,0.794731,0.799339,0.967646,0.918047,0.967014,0.835634,0.595369
8,"CTLS, REC",balanced_hist_deep,0.797834,0.797273,0.966355,0.918754,0.967001,0.832641,0.592178
9,"STLS, REC",balanced_hist_deep,0.790000,0.787634,0.963949,0.911890,0.967961,0.815763,0.579179
3,REC,balanced_hist_deep,0.794656,0.787445,0.963088,0.909669,0.967334,0.816632,0.578369
5,"FLOW, STLS",balanced_hist_deep,0.789942,0.786397,0.964687,0.911116,0.964180,0.821005,0.574005
13,"CTLS, STLS, REC",balanced_hist_shallow,0.794117,0.784954,0.966934,0.919144,0.968033,0.809153,0.577677
0,FLOW,balanced_hist_deep,0.778341,0.783665,0.959993,0.901198,0.952473,0.819612,0.578911


## System Category

This table isolates how well each feature set classifies the `system` category. A high `SystemF1` means the supervised classifier can separate system traffic cleanly from both malware and residual unknown traffic.


In [9]:
system_df = results_df.sort_values("SystemF1", ascending=False)
display(system_df)

,FeatureSet,BestConfig,ValMacroF1,MacroF1,MacroAucScore,MacroApScore,SystemF1,MalwareF1,UnknownF1
13,"CTLS, STLS, REC",balanced_hist_shallow,0.794117,0.784954,0.966934,0.919144,0.968033,0.809153,0.577677
9,"STLS, REC",balanced_hist_deep,0.790000,0.787634,0.963949,0.911890,0.967961,0.815763,0.579179
11,"FLOW, CTLS, REC",balanced_hist_deep,0.804478,0.824456,0.969970,0.925409,0.967575,0.865118,0.640675
3,REC,balanced_hist_deep,0.794656,0.787445,0.963088,0.909669,0.967334,0.816632,0.578369
6,"FLOW, REC",balanced_hist_deep,0.795551,0.800526,0.966993,0.916310,0.967032,0.837176,0.597370
12,"FLOW, STLS, REC",balanced_hist_deep,0.794731,0.799339,0.967646,0.918047,0.967014,0.835634,0.595369
8,"CTLS, REC",balanced_hist_deep,0.797834,0.797273,0.966355,0.918754,0.967001,0.832641,0.592178
14,"FLOW, CTLS, STLS, REC",balanced_hist_deep,0.804776,0.824986,0.971410,0.927685,0.966628,0.864503,0.643828
10,"FLOW, CTLS, STLS",balanced_hist_deep,0.796404,0.780686,0.965497,0.915258,0.964703,0.810669,0.566685
5,"FLOW, STLS",balanced_hist_deep,0.789942,0.786397,0.964687,0.911116,0.964180,0.821005,0.574005


## Malware Category

This section focuses on the `malware` class. Strong `MalwareF1` values indicate that the selected feature family helps XGBoost carve out malware traffic as a distinct supervised class rather than only separating it from one chosen normal profile.


In [10]:
malware_df = results_df.sort_values("MalwareF1", ascending=False)
display(malware_df)


,FeatureSet,BestConfig,ValMacroF1,MacroF1,MacroAucScore,MacroApScore,SystemF1,MalwareF1,UnknownF1
11,"FLOW, CTLS, REC",balanced_hist_deep,0.804478,0.824456,0.969970,0.925409,0.967575,0.865118,0.640675
14,"FLOW, CTLS, STLS, REC",balanced_hist_deep,0.804776,0.824986,0.971410,0.927685,0.966628,0.864503,0.643828
6,"FLOW, REC",balanced_hist_deep,0.795551,0.800526,0.966993,0.916310,0.967032,0.837176,0.597370
12,"FLOW, STLS, REC",balanced_hist_deep,0.794731,0.799339,0.967646,0.918047,0.967014,0.835634,0.595369
8,"CTLS, REC",balanced_hist_deep,0.797834,0.797273,0.966355,0.918754,0.967001,0.832641,0.592178
5,"FLOW, STLS",balanced_hist_deep,0.789942,0.786397,0.964687,0.911116,0.964180,0.821005,0.574005
0,FLOW,balanced_hist_deep,0.778341,0.783665,0.959993,0.901198,0.952473,0.819612,0.578911
3,REC,balanced_hist_deep,0.794656,0.787445,0.963088,0.909669,0.967334,0.816632,0.578369
9,"STLS, REC",balanced_hist_deep,0.790000,0.787634,0.963949,0.911890,0.967961,0.815763,0.579179
4,"FLOW, CTLS",balanced_hist_deep,0.791323,0.778054,0.962151,0.907716,0.961592,0.812584,0.559985


## Unknown Category

The `unknown` label remains heterogeneous, so this table should still be interpreted with care. `UnknownF1` shows how well the classifier treats that residual traffic bucket as its own class instead of collapsing it into `system` or `malware`.


In [11]:
unknown_df = results_df.sort_values("UnknownF1", ascending=False)
display(unknown_df)

,FeatureSet,BestConfig,ValMacroF1,MacroF1,MacroAucScore,MacroApScore,SystemF1,MalwareF1,UnknownF1
14,"FLOW, CTLS, STLS, REC",balanced_hist_deep,0.804776,0.824986,0.971410,0.927685,0.966628,0.864503,0.643828
11,"FLOW, CTLS, REC",balanced_hist_deep,0.804478,0.824456,0.969970,0.925409,0.967575,0.865118,0.640675
6,"FLOW, REC",balanced_hist_deep,0.795551,0.800526,0.966993,0.916310,0.967032,0.837176,0.597370
12,"FLOW, STLS, REC",balanced_hist_deep,0.794731,0.799339,0.967646,0.918047,0.967014,0.835634,0.595369
8,"CTLS, REC",balanced_hist_deep,0.797834,0.797273,0.966355,0.918754,0.967001,0.832641,0.592178
9,"STLS, REC",balanced_hist_deep,0.790000,0.787634,0.963949,0.911890,0.967961,0.815763,0.579179
0,FLOW,balanced_hist_deep,0.778341,0.783665,0.959993,0.901198,0.952473,0.819612,0.578911
3,REC,balanced_hist_deep,0.794656,0.787445,0.963088,0.909669,0.967334,0.816632,0.578369
13,"CTLS, STLS, REC",balanced_hist_shallow,0.794117,0.784954,0.966934,0.919144,0.968033,0.809153,0.577677
5,"FLOW, STLS",balanced_hist_deep,0.789942,0.786397,0.964687,0.911116,0.964180,0.821005,0.574005


## SHAP Interpretation

This section retrains the best overall XGBoost configuration and visualizes SHAP values for the `malware` class on a sample of the held-out test split. The goal is to show which transformed features most strongly push predictions toward the malware class in the best-performing supervised model.


In [ ]:
import matplotlib.pyplot as plt
import shap

def expand_feature_groups(group_combo):
    selected_features = []
    for group_name in group_combo:
        selected_features.extend(feature_groups[group_name])
    return selected_features

def extract_class_shap_values(shap_values, class_index):
    values = getattr(shap_values, "values", shap_values)
    if isinstance(values, list):
        return np.asarray(values[class_index])

    values = np.asarray(values)
    if values.ndim == 3:
        return values[:, :, class_index]
    if values.ndim == 2:
        return values

    raise ValueError(f"Unsupported SHAP value shape: {values.shape}")

best_row = overall_df.iloc[0]
best_group_combo = tuple(best_row["FeatureSet"].split(", "))
best_feature_set = expand_feature_groups(best_group_combo)

x_train_best = select_feature_set(preprocessor.transform(df_train), best_feature_set)
x_test_best = select_feature_set(preprocessor.transform(df_test), best_feature_set)
y_train_best = encode_labels(df_train_label["category"])

best_model = XGBoostBaseline(XGB_CONFIG_CANDIDATES[best_row["BestConfig"]])
best_model.fit(x_train_best, y_train_best)

background = x_train_best.sample(n=min(1000, len(x_train_best)), random_state=42)
explain_data = x_test_best.sample(n=min(2000, len(x_test_best)), random_state=42)

explainer = best_model.build_shap_explainer(background)
shap_values = best_model.explain(explain_data, explainer=explainer)

class_name = "malware"
class_index = CLASS_TO_INDEX[class_name]
class_shap_values = extract_class_shap_values(shap_values, class_index)

print(
    f"SHAP summary for feature set={best_row['FeatureSet']} "
    f"with config={best_row['BestConfig']} on class={class_name}."
)

plt.figure(figsize=(12, 8))
shap.summary_plot(
    class_shap_values,
    explain_data,
    feature_names=explain_data.columns.tolist(),
    show=False,
)
plt.title(f"SHAP Summary for '{class_name}' Class")
plt.tight_layout()
plt.show()


/home/rysavy/devel/GitHub/AutoFedProfile/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
 40%|========            | 2398/6000 [04:57<07:26]       